# 03 — Model Evaluation
## DermaVision · EfficientNet-B3 · Exp 07 (Optuna FINAL)

This notebook evaluates the best trained model on the held-out validation set.

| | |
|---|---|
| **Model** | EfficientNet-B3 (timm) |
| **Checkpoint** | `checkpoints/exp07-optuna-FINAL_best.pt` |
| **Val set** | 2 003 images — stratified 20% split, fixed `SEED=42` |
| **Primary metric** | AUC-ROC (macro OvR) |
| **Known result** | AUC = **0.9776**, Acc = 0.854, F1-macro = 0.796 |

### Contents
1. Setup & model loading  
2. Run inference on full validation set  
3. Per-class metrics table  
4. Confusion matrix  
5. ROC curves (one per class + macro average)  
6. Precision-Recall curves  
7. Worst predictions (high-confidence errors)  
8. Grad-CAM visualisations


In [1]:

# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Add repo root to path so `src` is importable
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from torch.utils.data import DataLoader
import torch.nn.functional as F

from src.utils.config import (
    CLASS_NAMES, LABEL_MAP, CKPT_DIR, IMG_SIZE, BATCH_SIZE
)
from src.data.dataset import build_dataframes, HAM10000Dataset
from src.data.preprocessing import get_val_transforms
from src.models.architecture import DermaVisionModel

# ── Nice plot style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
})
PALETTE = sns.color_palette("tab10", len(CLASS_NAMES))
CLASS_FULL = [LABEL_MAP[c] for c in CLASS_NAMES]

print(f"Repo root : {REPO_ROOT}")
print(f"Classes   : {CLASS_NAMES}")
print(f"Device    : ", end="")

# Auto-detect device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(DEVICE)


Repo root : /Users/bidoun/Documents/projet_fed
Classes   : ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
Device    : mps


## 2 — Load Model & Validation Set

In [2]:

# ── Load checkpoint ───────────────────────────────────────────────────────────
CKPT_NAME = "exp07-optuna-FINAL_best.pt"
ckpt_path  = CKPT_DIR / CKPT_NAME
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

model = DermaVisionModel(
    backbone="efficientnet_b3",
    num_classes=len(CLASS_NAMES),
    pretrained=False,
    dropout=0.229,          # Optuna best value
)
state_dict = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()
print(f"✓ Loaded {CKPT_NAME}")

# ── Build validation DataLoader ───────────────────────────────────────────────
_, val_df = build_dataframes()
val_ds    = HAM10000Dataset(val_df, images_dir=REPO_ROOT / "data/HAM10000/ISIC2018_Task3_Training_Input",
                             transform=get_val_transforms())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=False)

print(f"✓ Validation set: {len(val_ds)} images")


✓ Loaded exp07-optuna-FINAL_best.pt
✓ Validation set: 2003 images


## 3 — Run Inference on Full Validation Set

In [ ]:

from tqdm.auto import tqdm

all_probs  = []   # (N, 7) softmax probabilities
all_preds  = []   # (N,)   argmax predictions
all_labels = []   # (N,)   ground-truth integer labels

with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs  = F.softmax(logits, dim=-1).cpu().numpy()
        preds  = probs.argmax(axis=1)
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(labels.numpy())

all_probs  = np.vstack(all_probs)    # (2003, 7)
all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

print(f"Inference done — {len(all_labels)} samples")
print(f"Predicted class distribution:")
for i, c in enumerate(CLASS_NAMES):
    n = (all_preds == i).sum()
    print(f"  {c:6s} → {n:4d}  ({n/len(all_preds)*100:.1f}%)")


Inference:  43%|████▎     | 27/63 [00:09<00:09,  3.84it/s]

## 4 — Per-Class Metrics Table

In [ ]:

from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize
import pandas as pd

# One-hot encode labels for AUC
all_labels_oh = label_binarize(all_labels, classes=list(range(len(CLASS_NAMES))))

# Classification report → DataFrame
report = classification_report(
    all_labels, all_preds,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
df_report = pd.DataFrame(report).T.loc[CLASS_NAMES, ["precision", "recall", "f1-score", "support"]]
df_report["support"] = df_report["support"].astype(int)

# Per-class AUC-ROC
per_auc = {}
for i, c in enumerate(CLASS_NAMES):
    per_auc[c] = roc_auc_score(all_labels_oh[:, i], all_probs[:, i])
df_report["AUC-ROC"] = pd.Series(per_auc)

# Macro & weighted rows
df_report.loc["macro avg"] = df_report.mean(numeric_only=True)
df_report.loc["macro avg", "support"] = len(all_labels)

# Full name column
full_names = {
    "AKIEC": "Actinic Keratosis",
    "BCC":   "Basal Cell Carcinoma",
    "BKL":   "Benign Keratosis",
    "DF":    "Dermatofibroma",
    "MEL":   "Melanoma",
    "NV":    "Melanocytic Nevi",
    "VASC":  "Vascular Lesion",
    "macro avg": "—",
}
df_report.insert(0, "Full Name", df_report.index.map(lambda x: full_names.get(x, "")))

print(df_report.to_string(float_format="{:.3f}".format))


## 5 — Confusion Matrix

In [ ]:

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ["Counts", "Normalised (row %)"],
    ["d",     ".2f"],
):
    im = sns.heatmap(
        data,
        annot=True, fmt=fmt,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cmap="Blues", linewidths=0.4,
        ax=ax,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True", fontsize=11)
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)

plt.suptitle("Confusion Matrix — Validation Set", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/confusion_matrix.png")


## 6 — ROC Curves (One-vs-Rest)

In [ ]:

from sklearn.metrics import roc_curve, auc

COLORS = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(9, 7))

fpr_all, tpr_all = [], []
for i, c in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_oh[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=COLORS[i], lw=1.8, label=f"{c} (AUC = {roc_auc:.3f})")
    fpr_all.append(fpr)
    tpr_all.append(tpr)

# Macro-average (interpolated)
all_fpr = np.unique(np.concatenate(fpr_all))
mean_tpr = np.zeros_like(all_fpr)
for i in range(len(CLASS_NAMES)):
    mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
mean_tpr /= len(CLASS_NAMES)
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, color="black", lw=2.5, linestyle="--",
        label=f"Macro avg (AUC = {macro_auc:.3f})")

ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle=":")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — One-vs-Rest", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/roc_curves.png")


## 7 — Precision-Recall Curves

In [ ]:

from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(9, 7))

for i, c in enumerate(CLASS_NAMES):
    prec, rec, _ = precision_recall_curve(all_labels_oh[:, i], all_probs[:, i])
    ap = average_precision_score(all_labels_oh[:, i], all_probs[:, i])
    ax.plot(rec, prec, color=COLORS[i], lw=1.8, label=f"{c} (AP = {ap:.3f})")

mean_ap = np.mean([
    average_precision_score(all_labels_oh[:, i], all_probs[:, i])
    for i in range(len(CLASS_NAMES))
])
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title(f"Precision-Recall Curves  (mAP = {mean_ap:.3f})", fontsize=13)
ax.legend(loc="upper right", fontsize=9)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/pr_curves.png")


## 8 — Worst Predictions (High-Confidence Errors)

In [ ]:

from PIL import Image

# Indices where model was wrong
wrong_mask = (all_preds != all_labels)
wrong_idx  = np.where(wrong_mask)[0]
wrong_conf = all_probs[wrong_idx, all_preds[wrong_idx]]   # confidence of wrong pred

# Sort by descending confidence → most egregious errors first
order       = np.argsort(-wrong_conf)
top_n       = min(20, len(order))
top_wrong   = wrong_idx[order[:top_n]]

# Build a grid 4 cols × 5 rows
ncols, nrows = 4, 5
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.5))

for ax_idx, sample_idx in enumerate(top_wrong):
    row, col = divmod(ax_idx, ncols)
    ax = axes[row][col]

    img_path = val_dataset.df.iloc[sample_idx]["image_path"]
    img      = Image.open(img_path).convert("RGB")
    true_cls = CLASS_NAMES[all_labels[sample_idx]]
    pred_cls = CLASS_NAMES[all_preds[sample_idx]]
    conf     = wrong_conf[order[ax_idx]] * 100

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(
        f"True: {true_cls}\nPred: {pred_cls} ({conf:.0f}%)",
        fontsize=8, color="red", pad=3
    )

# Hide unused axes
for ax_idx in range(top_n, nrows * ncols):
    row, col = divmod(ax_idx, ncols)
    axes[row][col].axis("off")

plt.suptitle("Top Worst Predictions (highest-confidence errors)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/worst_predictions.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved → outputs/worst_predictions.png")


## 9 — Grad-CAM Visualisations

In [ ]:

import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Target layer = last conv block of EfficientNet-B3
target_layer = [model.base.blocks[-1]]

# Pick 8 samples: one correctly-predicted from each of the 7 classes + 1 wrong
chosen = []
for cls_idx in range(len(CLASS_NAMES)):
    correct_idx = np.where((all_labels == cls_idx) & (all_preds == cls_idx))[0]
    if len(correct_idx):
        chosen.append((correct_idx[0], True))  # (sample_idx, is_correct)
if wrong_idx.size > 0:
    chosen.append((wrong_idx[0], False))

ncols, nrows = 4, 2
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.5))
axes_flat = axes.flat

val_transforms_nocrop = get_val_transforms()

with GradCAM(model=model, target_layers=target_layer) as cam:
    for plot_idx, (sample_idx, correct) in enumerate(chosen[:ncols * nrows]):
        img_path  = val_dataset.df.iloc[sample_idx]["image_path"]
        raw_img   = np.array(Image.open(img_path).convert("RGB").resize((300, 300)))
        rgb_float = raw_img.astype(np.float32) / 255.0

        # Prepare tensor
        tensor    = val_transforms_nocrop(Image.fromarray(raw_img)).unsqueeze(0).to(DEVICE)
        cls_idx   = int(all_preds[sample_idx])
        targets   = [ClassifierOutputTarget(cls_idx)]

        grayscale = cam(input_tensor=tensor, targets=targets)[0]  # (H, W)
        heatmap   = show_cam_on_image(
            cv2.resize(rgb_float, (grayscale.shape[1], grayscale.shape[0])),
            grayscale, use_rgb=True
        )

        ax = next(axes_flat)
        ax.imshow(heatmap)
        ax.axis("off")
        true_name = CLASS_NAMES[all_labels[sample_idx]]
        pred_name = CLASS_NAMES[all_preds[sample_idx]]
        color     = "green" if correct else "red"
        conf_val  = all_probs[sample_idx, cls_idx] * 100
        ax.set_title(
            f"True: {true_name}\nPred: {pred_name} ({conf_val:.0f}%)",
            fontsize=8, color=color, pad=3
        )

# Hide remaining axes
for ax in axes_flat:
    ax.axis("off")

plt.suptitle("Grad-CAM Heatmaps (green = correct, red = wrong)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/gradcam.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved → outputs/gradcam.png")


## 10 — Summary & Final Remarks

In [ ]:

from sklearn.metrics import accuracy_score, f1_score

acc     = accuracy_score(all_labels, all_preds)
f1_mac  = f1_score(all_labels, all_preds, average="macro",    zero_division=0)
f1_wtd  = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
auc_mac = roc_auc_score(all_labels_oh, all_probs, average="macro")

print("=" * 50)
print("  FINAL EVALUATION — Validation Set (n=2003)")
print("=" * 50)
print(f"  Accuracy       : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  F1  (macro)    : {f1_mac:.4f}")
print(f"  F1  (weighted) : {f1_wtd:.4f}")
print(f"  AUC-ROC (macro): {auc_mac:.4f}")
print("=" * 50)
print("\nAll plots saved to outputs/")
